<a href="https://colab.research.google.com/github/FlavioMartelli/Pokemon-Battle/blob/Giulia/prova_fds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

In [5]:
# ----------------------------
# 1. COSTANTI
# ----------------------------

if os.path.exists("/content/train.jsonl"):
    base_path = "/content/"
else:
    base_path = "./"

print("Percorso dati:", base_path)

type_df = pd.read_csv(
    os.path.join(base_path, "type_chart_gen2.csv")
)

# Normalizza i nomi dei tipi nella type chart.
type_df["attacking_type"] = (
    type_df["attacking_type"].astype(str).str.lower()
)
type_df["defending_type"] = (
    type_df["defending_type"].astype(str).str.lower()
)

TYPE_MULTIPLIER = {
    (row.attacking_type, row.defending_type): float(row.multiplier)
    for row in type_df.itertuples()
}

PHYSICAL_TYPES = {
    "normal", "fighting", "flying", "poison", "ground",
    "rock", "bug", "ghost", "steel"
}
SPECIAL_TYPES = {
    "fire", "water", "grass", "electric",
    "psychic", "ice", "dragon", "dark"
}

STAT_KEYS = [
    "base_hp", "base_atk", "base_def",
    "base_spa", "base_spd", "base_spe"
]

STAT_SHORT = ["hp", "atk", "def", "spa", "spd", "spe"]

STATUS_LIST = ["slp", "frz", "par", "brn", "tox"]
IMPORTANT_ITEMS = [
    "leftovers", "thickclub", "lightball", "berserkgene",
    "mintberry", "miracleberry", "bitterberry", "przcureberry",
    "quickclaw", "kingsrock", "focusband", "metalpowder",
    "stick"
]

STRATEGIC_MOVES = {
    "curse", "rest", "sleeptalk", "spikes", "rapidspin",
    "explosion", "selfdestruct", "self-destruct", "toxic",
    "thunderwave", "stunspore", "roar", "whirlwind",
    "reflect", "lightscreen", "light-screen", "protect",
    "substitute", "bellydrum", "swordsdance", "recover",
    "softboiled", "moonlight", "morningsun", "synthesis",
    "perishsong", "encore", "batonpass", "haze",
    "sandstorm", "raindance", "sunnyday"
}

# Indicatori strategici basati su specie comunemente rilevanti in GSC OU.
OU_SPECIES = {
    "snorlax", "zapdos", "raikou", "skarmory", "cloyster",
    "tyranitar", "marowak", "umbreon", "suicune", "exeggutor",
    "machamp", "nidoking", "heracross", "steelix", "starmie",
    "forretress", "blissey", "miltank", "gengar"
}


# ----------------------------
# 2. FUNZIONI DI SUPPORTO
# ----------------------------

def safe_float(value, default=0.0):
    try:
        if value is None:
            return default
        value = float(value)
        if not np.isfinite(value):
            return default
        return value
    except (TypeError, ValueError):
        return default


def clean_type_list(types):
    if not isinstance(types, list):
        return []
    return [
        str(t).lower()
        for t in types
        if t is not None and str(t).lower() not in ("notype", "")
    ]


def get_type_multiplier(attack_type, defender_types):
    """Moltiplicatore Gen 2 per una mossa contro uno o due tipi."""
    attack_type = str(attack_type).lower()
    multiplier = 1.0

    for defender_type in clean_type_list(defender_types):
        multiplier *= TYPE_MULTIPLIER.get(
            (attack_type, defender_type), 1.0
        )

    return multiplier


def get_bst(pokemon):
    return sum(safe_float(pokemon.get(k, 0)) for k in STAT_KEYS)


def get_state(turn, player):
    state = turn.get(f"{player}_pokemon_state", {})
    return state if isinstance(state, dict) else {}


def get_move(turn, player):
    move = turn.get(f"{player}_move_details")
    return move if isinstance(move, dict) else None


def get_boost_sum(state):
    boosts = state.get("boosts", {})
    if not isinstance(boosts, dict):
        return 0.0
    return sum(safe_float(v) for v in boosts.values())


def hp_value(state):
    return safe_float(state.get("hp_pct", 0.0), 0.0)


def get_name(state):
    return str(state.get("name", "")).lower()


def count_unique_fainted(timeline, player):
    names = set()
    for turn in timeline:
        state = get_state(turn, player)
        if state.get("status") == "fnt" and get_name(state):
            names.add(get_name(state))
    return len(names)


def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                records.append(json.loads(line))
    return records


# ----------------------------
# 2b. DATABASE SPECIE + SQUADRA P2 OSSERVATA
# ----------------------------

CACHE_DIR = os.path.join(base_path, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

SPECIES_DB = {}


def build_species_db():
    """Dizionario specie -> tipi e stat base, da P1 team e P2 lead (train+test)."""
    cache_path = os.path.join(CACHE_DIR, "species_db.pkl")
    if os.path.exists(cache_path):
        return pd.read_pickle(cache_path)

    db = {}
    for split in ["train", "test"]:
        for battle in load_jsonl(os.path.join(base_path, f"{split}.jsonl")):
            members = list(battle.get("p1_team_details", []) or [])
            lead = battle.get("p2_lead_details")
            if isinstance(lead, dict) and lead:
                members.append(lead)
            for p in members:
                name = str(p.get("name", "")).lower()
                if name and name not in db:
                    db[name] = {
                        "name": name,
                        "types": p.get("types", []),
                        **{k: p.get(k, 0) for k in STAT_KEYS},
                        "item": "unknown",  # l'item di P2 non è noto
                    }
    pd.to_pickle(db, cache_path)
    return db


def get_p2_observed_team(timeline, p2_lead):
    """Squadra di P2 ricostruita: lead + Pokémon comparsi nella timeline."""
    lead_name = str(p2_lead.get("name", "")).lower() if p2_lead else ""
    names = [lead_name] if lead_name else []
    for t in timeline:
        n = get_name(get_state(t, "p2"))
        if n and n not in names:
            names.append(n)

    team = []
    for n in names:
        if n == lead_name:
            team.append(p2_lead)
        elif n in SPECIES_DB:
            team.append(SPECIES_DB[n])
    return team


def avg_best_mult(move_types, defenders):
    """Media, sui difensori, del miglior moltiplicatore tra i tipi di mossa."""
    vals = []
    for d in defenders:
        d_types = clean_type_list(d.get("types", []))
        mults = [get_type_multiplier(t, d_types) for t in move_types]
        vals.append(max(mults) if mults else 1.0)
    return float(np.mean(vals)) if vals else 1.0


def add_team_features(feat, team, prefix):
    """Statistiche, tipi, strumenti e ruoli della squadra conosciuta."""
    team = team if isinstance(team, list) else []

    feat[f"{prefix}_team_size"] = len(team)

    for stat_key, short in zip(STAT_KEYS, STAT_SHORT):
        values = [safe_float(p.get(stat_key, 0)) for p in team]
        feat[f"{prefix}_sum_{short}"] = sum(values)
        feat[f"{prefix}_mean_{short}"] = np.mean(values) if values else 0
        feat[f"{prefix}_max_{short}"] = max(values) if values else 0
        feat[f"{prefix}_min_{short}"] = min(values) if values else 0
        feat[f"{prefix}_std_{short}"] = np.std(values) if values else 0

    bsts = [get_bst(p) for p in team]
    feat[f"{prefix}_bst_sum"] = sum(bsts)
    feat[f"{prefix}_bst_mean"] = np.mean(bsts) if bsts else 0
    feat[f"{prefix}_bst_max"] = max(bsts) if bsts else 0

    # Profili statistici della squadra.
    atk = [safe_float(p.get("base_atk", 0)) for p in team]
    spa = [safe_float(p.get("base_spa", 0)) for p in team]
    deff = [safe_float(p.get("base_def", 0)) for p in team]
    spd = [safe_float(p.get("base_spd", 0)) for p in team]
    spe = [safe_float(p.get("base_spe", 0)) for p in team]
    hp = [safe_float(p.get("base_hp", 0)) for p in team]
    feat[f"{prefix}_physical_offense_mean"] = float(np.mean(atk)) if atk else 0
    feat[f"{prefix}_special_offense_mean"] = float(np.mean(spa)) if spa else 0
    feat[f"{prefix}_physical_bulk_mean"] = float(np.mean(deff)) if deff else 0
    feat[f"{prefix}_special_bulk_mean"] = float(np.mean(spd)) if spd else 0
    feat[f"{prefix}_speed_mean"] = float(np.mean(spe)) if spe else 0
    feat[f"{prefix}_speed_max"] = max(spe) if spe else 0
    feat[f"{prefix}_fast_count_100"] = sum(v >= 100 for v in spe)
    feat[f"{prefix}_fast_count_80"] = sum(v >= 80 for v in spe)
    feat[f"{prefix}_physical_bias"] = (
        float(np.mean(atk)) - float(np.mean(spa)) if atk and spa else 0
    )
    feat[f"{prefix}_bulk_mean"] = (
        (float(np.mean(hp)) + float(np.mean(deff)) + float(np.mean(spd))) / 3
        if hp and deff and spd else 0
    )

    # Tipi presenti nella squadra.
    all_types = []
    for pokemon in team:
        all_types.extend(clean_type_list(pokemon.get("types", [])))

    for pokemon_type in sorted(
        PHYSICAL_TYPES | SPECIAL_TYPES
    ):
        feat[f"{prefix}_type_count_{pokemon_type}"] = (
            all_types.count(pokemon_type)
        )

    feat[f"{prefix}_unique_types"] = len(set(all_types))
    feat[f"{prefix}_has_steel"] = int("steel" in all_types)
    feat[f"{prefix}_has_dark"] = int("dark" in all_types)
    feat[f"{prefix}_has_ghost"] = int("ghost" in all_types)
    feat[f"{prefix}_has_flying"] = int("flying" in all_types)

    # Strumenti.
    items = [
        str(p.get("item", "") or "").lower()
        for p in team
    ]
    for item in IMPORTANT_ITEMS:
        feat[f"{prefix}_item_{item}"] = items.count(item)

    feat[f"{prefix}_no_item_count"] = sum(
        item in ("", "noitem", "none") for item in items
    )

    # Pokémon e combinazioni note del metagame.
    names = [str(p.get("name", "")).lower() for p in team]
    for species in sorted(OU_SPECIES):
        feat[f"{prefix}_species_{species}"] = int(species in names)

    feat[f"{prefix}_has_marowak_thickclub"] = int(
        any(
            str(p.get("name", "")).lower() == "marowak"
            and str(p.get("item", "")).lower() == "thickclub"
            for p in team
        )
    )
    feat[f"{prefix}_has_pikachu_lightball"] = int(
        any(
            str(p.get("name", "")).lower() == "pikachu"
            and str(p.get("item", "")).lower() == "lightball"
            for p in team
        )
    )


# ----------------------------
# 3. FEATURE ENGINEERING
# ----------------------------

def extract_features(data):
    rows = []

    for battle in data:
        feat = {}
        feat["battle_id"] = battle.get("battle_id")

        p1_team = battle.get("p1_team_details", [])
        if not isinstance(p1_team, list):
            p1_team = []

        p1_lead = p1_team[0] if p1_team else {}
        p2_lead = battle.get("p2_lead_details", {})
        if not isinstance(p2_lead, dict):
            p2_lead = {}

        # ----- Squadre e statistiche base -----
        add_team_features(feat, p1_team, "p1")

        # Per P2 il lead è noto con certezza.
        add_team_features(feat, [p2_lead] if p2_lead else [], "p2lead")

        # Statistiche e differenze del lead.
        for stat_key, short in zip(STAT_KEYS, STAT_SHORT):
            p1_value = safe_float(p1_lead.get(stat_key, 0))
            p2_value = safe_float(p2_lead.get(stat_key, 0))
            feat[f"lead_p1_{short}"] = p1_value
            feat[f"lead_p2_{short}"] = p2_value
            feat[f"lead_diff_{short}"] = p1_value - p2_value

        feat["lead_p1_bst"] = get_bst(p1_lead)
        feat["lead_p2_bst"] = get_bst(p2_lead)
        feat["lead_bst_diff"] = (
            feat["lead_p1_bst"] - feat["lead_p2_bst"]
        )
        feat["lead_p1_faster"] = int(
            safe_float(p1_lead.get("base_spe", 0))
            > safe_float(p2_lead.get("base_spe", 0))
        )
        for stat in ["base_atk", "base_def", "base_spa", "base_spd", "base_hp", "base_spe"]:
            a = safe_float(p1_lead.get(stat, 0))
            b = safe_float(p2_lead.get(stat, 0))
            feat[f"lead_ratio_{stat}"] = a / (b + 1.0)
        feat["lead_p1_physical_bias"] = (
            safe_float(p1_lead.get("base_atk", 0)) -
            safe_float(p1_lead.get("base_spa", 0))
        )
        feat["lead_p2_physical_bias"] = (
            safe_float(p2_lead.get("base_atk", 0)) -
            safe_float(p2_lead.get("base_spa", 0))
        )

        # Matchup difensivo dei lead (proxy basata sui tipi).
        p1_types = clean_type_list(p1_lead.get("types", []))
        p2_types = clean_type_list(p2_lead.get("types", []))

        for attack_type in sorted(PHYSICAL_TYPES | SPECIAL_TYPES):
            feat[f"p1lead_def_mult_from_{attack_type}"] = (
                get_type_multiplier(attack_type, p1_types)
            )
            feat[f"p2lead_def_mult_from_{attack_type}"] = (
                get_type_multiplier(attack_type, p2_types)
            )

        p1_best_type_matchup = [
            get_type_multiplier(t, p2_types) for t in p1_types
        ]
        p2_best_type_matchup = [
            get_type_multiplier(t, p1_types) for t in p2_types
        ]
        feat["lead_best_type_attack_p1"] = (
            max(p1_best_type_matchup) if p1_best_type_matchup else 1.0
        )
        feat["lead_best_type_attack_p2"] = (
            max(p2_best_type_matchup) if p2_best_type_matchup else 1.0
        )

        # ----- Timeline -----
        timeline = battle.get("battle_timeline", [])
        if not isinstance(timeline, list):
            timeline = []

        feat["num_turns"] = len(timeline)

        p1_states = [get_state(t, "p1") for t in timeline]
        p2_states = [get_state(t, "p2") for t in timeline]

        p1_hp = [hp_value(s) for s in p1_states]
        p2_hp = [hp_value(s) for s in p2_states]
        hp_diffs = [a - b for a, b in zip(p1_hp, p2_hp)]

        # Snapshot HP a turni prefissati.
        for checkpoint in [1, 3, 5, 10, 15, 20, 25, 30]:
            idx = min(checkpoint - 1, len(timeline) - 1)
            if idx >= 0:
                feat[f"hp_diff_turn_{checkpoint}"] = hp_diffs[idx]
                feat[f"p1_hp_turn_{checkpoint}"] = p1_hp[idx]
                feat[f"p2_hp_turn_{checkpoint}"] = p2_hp[idx]
            else:
                feat[f"hp_diff_turn_{checkpoint}"] = 0
                feat[f"p1_hp_turn_{checkpoint}"] = 0
                feat[f"p2_hp_turn_{checkpoint}"] = 0

        if timeline:
            feat["p1_last_active_hp"] = p1_hp[-1]
            feat["p2_last_active_hp"] = p2_hp[-1]
            feat["last_active_hp_diff"] = hp_diffs[-1]
            feat["p1_last_active_fainted"] = int(
                p1_states[-1].get("status") == "fnt"
            )
            feat["p2_last_active_fainted"] = int(
                p2_states[-1].get("status") == "fnt"
            )
        else:
            for key in [
                "p1_last_active_hp", "p2_last_active_hp",
                "last_active_hp_diff", "p1_last_active_fainted",
                "p2_last_active_fainted"
            ]:
                feat[key] = 0

        # HP trend: regressione lineare sul differenziale osservato.
        if len(hp_diffs) >= 2:
            feat["hp_diff_trend"] = float(
                np.polyfit(np.arange(len(hp_diffs)), hp_diffs, 1)[0]
            )
            feat["hp_diff_mean"] = float(np.mean(hp_diffs))
            feat["hp_diff_min"] = float(np.min(hp_diffs))
            feat["hp_diff_max"] = float(np.max(hp_diffs))
            feat["hp_diff_std"] = float(np.std(hp_diffs))
        else:
            feat["hp_diff_trend"] = 0
            feat["hp_diff_mean"] = 0
            feat["hp_diff_min"] = 0
            feat["hp_diff_max"] = 0
            feat["hp_diff_std"] = 0

        feat["p1_active_hp_change"] = (
            p1_hp[-1] - p1_hp[0] if len(p1_hp) >= 2 else 0
        )
        feat["p2_active_hp_change"] = (
            p2_hp[-1] - p2_hp[0] if len(p2_hp) >= 2 else 0
        )

        # Fasi della lotta.
        for phase, start, end in [
            ("early", 0, 5),
            ("mid", 5, 15),
            ("late", 15, 30)
        ]:
            diffs = hp_diffs[start:end]
            feat[f"{phase}_hp_diff_mean"] = (
                float(np.mean(diffs)) if diffs else 0
            )
            feat[f"{phase}_hp_diff_last"] = (
                float(diffs[-1]) if diffs else 0
            )

        # KO distinti osservati.
        feat["p1_observed_fainted_count"] = count_unique_fainted(
            timeline, "p1"
        )
        feat["p2_observed_fainted_count"] = count_unique_fainted(
            timeline, "p2"
        )
        feat["observed_fainted_diff"] = (
            feat["p2_observed_fainted_count"]
            - feat["p1_observed_fainted_count"]
        )

        # Pokémon attivi distinti e cambi.
        p1_names = [get_name(s) for s in p1_states if get_name(s)]
        p2_names = [get_name(s) for s in p2_states if get_name(s)]

        feat["p1_distinct_active_pokemon"] = len(set(p1_names))
        feat["p2_distinct_active_pokemon"] = len(set(p2_names))

        feat["p1_switches_observed"] = sum(
            a != b for a, b in zip(p1_names[:-1], p1_names[1:])
        )
        feat["p2_switches_observed"] = sum(
            a != b for a, b in zip(p2_names[:-1], p2_names[1:])
        )

        # Status e boost.
        for player, states in [("p1", p1_states), ("p2", p2_states)]:
            for status in STATUS_LIST:
                status_states = [
                    s for s in states if s.get("status") == status
                ]
                feat[f"{player}_{status}_turns"] = len(status_states)
                feat[f"{player}_{status}_pokemon_count"] = len({
                    get_name(s) for s in status_states if get_name(s)
                })

            feat[f"{player}_boost_sum_mean"] = (
                np.mean([get_boost_sum(s) for s in states])
                if states else 0
            )
            feat[f"{player}_boost_sum_max"] = (
                max([get_boost_sum(s) for s in states], default=0)
            )
            feat[f"{player}_boost_sum_last"] = (
                get_boost_sum(states[-1]) if states else 0
            )

        feat["boost_last_diff"] = (
            feat["p1_boost_sum_last"] - feat["p2_boost_sum_last"]
        )

        # Mosse osservate.
        p1_moves = [get_move(t, "p1") for t in timeline]
        p2_moves = [get_move(t, "p2") for t in timeline]
        p1_moves = [m for m in p1_moves if m is not None]
        p2_moves = [m for m in p2_moves if m is not None]

        for player, moves in [("p1", p1_moves), ("p2", p2_moves)]:
            names = [
                str(m.get("name", "")).lower().replace(" ", "")
                for m in moves
            ]
            powers = [
                safe_float(m.get("base_power", 0))
                for m in moves
            ]

            feat[f"{player}_move_count"] = len(moves)
            feat[f"{player}_avg_move_power"] = (
                float(np.mean(powers)) if powers else 0
            )
            feat[f"{player}_max_move_power"] = (
                max(powers) if powers else 0
            )
            feat[f"{player}_high_power_moves"] = sum(
                power >= 100 for power in powers
            )

            for move_name in sorted(STRATEGIC_MOVES):
                normalized = move_name.replace("-", "").replace(" ", "")
                feat[f"{player}_move_{normalized}"] = sum(
                    n == normalized for n in names
                )

            feat[f"{player}_explosion_count"] = sum(
                n in ("explosion", "selfdestruct") for n in names
            )
            feat[f"{player}_rest_count"] = names.count("rest")
            feat[f"{player}_sleeptalk_count"] = names.count("sleeptalk")
            feat[f"{player}_spikes_count"] = names.count("spikes")
            feat[f"{player}_rapidspin_count"] = names.count("rapidspin")
            feat[f"{player}_curse_count"] = names.count("curse")
            feat[f"{player}_toxic_count"] = names.count("toxic")
            feat[f"{player}_thunderwave_count"] = names.count("thunderwave")

            # Gen 2: fisico/speciale dipende dal TIPO.
            physical = 0
            special = 0
            status_moves = 0
            for move in moves:
                category = str(move.get("category", "")).lower()
                move_type = str(move.get("type", "")).lower()
                power = safe_float(move.get("base_power", 0))

                if category in ("status", "non-damaging") or power <= 0:
                    status_moves += 1
                elif move_type in PHYSICAL_TYPES:
                    physical += 1
                elif move_type in SPECIAL_TYPES:
                    special += 1

            feat[f"{player}_physical_moves"] = physical
            feat[f"{player}_special_moves"] = special
            feat[f"{player}_status_moves"] = status_moves

        # RestTalk osservato.
        feat["p1_resttalk_combo"] = int(
            feat["p1_rest_count"] > 0
            and feat["p1_sleeptalk_count"] > 0
        )
        feat["p2_resttalk_combo"] = int(
            feat["p2_rest_count"] > 0
            and feat["p2_sleeptalk_count"] > 0
        )

        # Copertura offensiva osservata (lead vs lead).
        p1_lead_move_types = {
            str(m.get("type", "")).lower()
            for m in p1_moves
            if str(m.get("type", "")).lower() in (PHYSICAL_TYPES | SPECIAL_TYPES)
        }
        p2_lead_move_types = {
            str(m.get("type", "")).lower()
            for m in p2_moves
            if str(m.get("type", "")).lower() in (PHYSICAL_TYPES | SPECIAL_TYPES)
        }

        feat["p1_move_type_coverage_vs_p2lead"] = max(
            [
                get_type_multiplier(t, p2_types)
                for t in p1_lead_move_types
            ] or [1.0]
        )
        feat["p2_move_type_coverage_vs_p1lead"] = max(
            [
                get_type_multiplier(t, p1_types)
                for t in p2_lead_move_types
            ] or [1.0]
        )

        # Archetipi dedotti dalle mosse osservate.
        p1_move_set = {
            str(m.get("name", "")).lower().replace(" ", "")
            for m in p1_moves
        }
        p2_move_set = {
            str(m.get("name", "")).lower().replace(" ", "")
            for m in p2_moves
        }

        for player, move_set in [
            ("p1", p1_move_set), ("p2", p2_move_set)
        ]:
            feat[f"{player}_observed_curse_rest"] = int(
                "curse" in move_set and "rest" in move_set
            )
            feat[f"{player}_observed_spikes_spin"] = int(
                "spikes" in move_set and "rapidspin" in move_set
            )
            feat[f"{player}_observed_screen"] = int(
                "reflect" in move_set
                or "lightscreen" in move_set
                or "light-screen" in move_set
            )

        # ----- NUOVO: squadra P2 osservata (lead + Pokémon rivelati) -----
        p2_team_obs = get_p2_observed_team(timeline, p2_lead)
        add_team_features(feat, p2_team_obs, "p2obs")
        feat["p2obs_revealed_count"] = len(p2_team_obs)

        # Differenze P1 (squadra intera) vs P2 (squadra osservata).
        for short in ["atk", "def", "spa", "spd", "spe", "hp"]:
            feat[f"obs_mean_diff_{short}"] = (
                feat[f"p1_mean_{short}"] - feat[f"p2obs_mean_{short}"]
            )
        feat["obs_bst_mean_diff"] = feat["p1_bst_mean"] - feat["p2obs_bst_mean"]
        feat["obs_speed_max_diff"] = feat["p1_speed_max"] - feat["p2obs_speed_max"]

        # Copertura delle mosse contro TUTTI i Pokémon rivelati / la squadra P1.
        p1_move_types_all = {
            str(m.get("type", "")).lower() for m in p1_moves
        } & (PHYSICAL_TYPES | SPECIAL_TYPES)
        p2_move_types_all = {
            str(m.get("type", "")).lower() for m in p2_moves
        } & (PHYSICAL_TYPES | SPECIAL_TYPES)

        feat["p1_moves_vs_p2obs_avg_best"] = avg_best_mult(
            p1_move_types_all, p2_team_obs
        )
        feat["p2_moves_vs_p1team_avg_best"] = avg_best_mult(
            p2_move_types_all, p1_team
        )
        feat["coverage_team_diff"] = (
            feat["p1_moves_vs_p2obs_avg_best"]
            - feat["p2_moves_vs_p1team_avg_best"]
        )

        # Debolezze di tipo di P1 rispetto ai tipi dei P2 rivelati.
        p2_obs_types = set()
        for p in p2_team_obs:
            p2_obs_types |= set(clean_type_list(p.get("types", [])))
        feat["p1_team_avg_mult_vs_p2obs_types"] = (
            float(np.mean([
                get_type_multiplier(t, p.get("types", []))
                for p in p1_team for t in p2_obs_types
            ])) if p1_team and p2_obs_types else 1.0
        )

        # Etichetta soltanto nel train.
        if "player_won" in battle:
            feat["player_won"] = int(bool(battle["player_won"]))

        rows.append(feat)

    return pd.DataFrame(rows).replace(
        [np.inf, -np.inf], np.nan
    ).fillna(0)


# ----------------------------
# CONFIGURAZIONE
# ----------------------------

QUICK = True              # True: 1 modello, 3 fold (per iterare sulle feature)
                          # False: XGB + Hist + LogReg, 5 fold, media semplice
FEATURE_VERSION = "v5"    # CAMBIALO ogni volta che modifichi extract_features
RANDOM_STATE = 42
N_SPLITS = 3 if QUICK else 5


# ----------------------------
# 4. CARICAMENTO DATI (con cache delle feature)
# ----------------------------

def load_features(split):
    cache_path = os.path.join(CACHE_DIR, f"{split}_features_{FEATURE_VERSION}.pkl")
    if os.path.exists(cache_path):
        print(f"Feature {split} caricate dalla cache: {cache_path}")
        return pd.read_pickle(cache_path)

    print(f"Estrazione feature {split}...")
    data = load_jsonl(os.path.join(base_path, f"{split}.jsonl"))
    df = extract_features(data)
    del data
    df.to_pickle(cache_path)
    return df


SPECIES_DB = build_species_db()
print("Specie nel database:", len(SPECIES_DB))

train_df = load_features("train")
test_df = load_features("test")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


# ----------------------------
# 5. MATRICE FEATURE
# ----------------------------

target = "player_won"
id_col = "battle_id"

features = [c for c in train_df.columns if c not in [target, id_col]]

X_train = train_df[features].copy()
y_train = train_df[target].astype(int).copy()
X_test = test_df.reindex(columns=features, fill_value=0).copy()

print("Numero feature:", len(features))
print("Distribuzione target:")
print(y_train.value_counts(normalize=True))


# ----------------------------
# 6. MODELLI
# ----------------------------

def make_xgb():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        XGBClassifier(
            n_estimators=700, max_depth=3, learning_rate=0.03,
            subsample=0.85, colsample_bytree=0.85,
            min_child_weight=5, reg_lambda=8, reg_alpha=0.2, gamma=0.05,
            eval_metric="logloss", objective="binary:logistic",
            n_jobs=-1, random_state=RANDOM_STATE
        )
    )


def make_hist():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        HistGradientBoostingClassifier(
            max_iter=400, learning_rate=0.03, max_leaf_nodes=15,
            l2_regularization=2.0, random_state=RANDOM_STATE
        )
    )


def make_logreg():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(
            C=0.5, max_iter=3000, solver="lbfgs",
            random_state=RANDOM_STATE
        )
    )


if QUICK:
    model_factories = {"XGBoost": make_xgb}
else:
    model_factories = {
        "XGBoost": make_xgb,
        "HistGradientBoosting": make_hist,
        "Logistic Regression": make_logreg,
    }

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
folds = list(skf.split(X_train, y_train))


def evaluate_model(model_name, model_factory):
    """OOF + media delle predizioni test sui fold, con cache su disco."""
    safe_name = model_name.replace(" ", "_")
    cache_path = os.path.join(
        CACHE_DIR, f"pred_{FEATURE_VERSION}_{safe_name}_k{N_SPLITS}.npz"
    )

    if os.path.exists(cache_path):
        print(f"  Predizioni {model_name} caricate dalla cache")
        saved = np.load(cache_path)
        return saved["oof"], saved["test"]

    oof = np.zeros(len(X_train), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
        model = model_factory()
        model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        valid_prob = model.predict_proba(X_train.iloc[va_idx])[:, 1]
        oof[va_idx] = valid_prob
        test_pred += model.predict_proba(X_test)[:, 1] / N_SPLITS

        fold_acc = accuracy_score(
            y_train.iloc[va_idx], (valid_prob >= 0.5).astype(int)
        )
        print(f"  Fold {fold}: accuracy = {fold_acc:.5f}")

    np.savez(cache_path, oof=oof, test=test_pred)
    return oof, test_pred


results = {}
oof_predictions = {}
test_predictions = {}

for name, factory in model_factories.items():
    print("\n" + "=" * 60 + "\nMODELLO:", name)
    oof, test_pred = evaluate_model(name, factory)
    results[name] = accuracy_score(y_train, (oof >= 0.5).astype(int))
    oof_predictions[name] = oof
    test_predictions[name] = test_pred
    print(f"  OOF accuracy {name}: {results[name]:.5f}")


# ----------------------------
# 7. ENSEMBLE: media semplice, soglia 0.5
# ----------------------------

print("\n========== RISULTATI ==========")
for name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name}: {score:.5f}")

if len(model_factories) > 1:
    ens_oof = np.mean([oof_predictions[n] for n in model_factories], axis=0)
    ens_test = np.mean([test_predictions[n] for n in model_factories], axis=0)
    ens_acc = accuracy_score(y_train, (ens_oof >= 0.5).astype(int))
    print(f"\nEnsemble (media semplice, soglia 0.5): {ens_acc:.5f}")
    final_test_predictions = ens_test
else:
    only = next(iter(model_factories))
    final_test_predictions = test_predictions[only]


# ----------------------------
# 8. SUBMISSION
# ----------------------------

sample_sub = pd.read_csv(os.path.join(base_path, "sample_submission.csv"))

preds_series = pd.Series(
    (final_test_predictions >= 0.5).astype(int),
    index=test_df[id_col]
)

submission = sample_sub[[id_col]].copy()
submission[target] = submission[id_col].map(preds_series).astype(int)

assert list(submission.columns) == ["battle_id", "player_won"]
assert len(submission) == len(test_df)
assert set(submission["player_won"].unique()).issubset({0, 1})

submission.to_csv("submission.csv", index=False)
print("\nSubmission creata: submission.csv", submission.shape)

Percorso dati: /content/
Specie nel database: 247
Estrazione feature train...
Estrazione feature test...
Train shape: (10000, 561)
Test shape: (5000, 560)
Numero feature: 559
Distribuzione target:
player_won
0    0.5
1    0.5
Name: proportion, dtype: float64

MODELLO: XGBoost
  Fold 1: accuracy = 0.64997
  Fold 2: accuracy = 0.66247
  Fold 3: accuracy = 0.65347
  OOF accuracy XGBoost: 0.65530

========== RISULTATI ==========
XGBoost: 0.65530

Submission creata: submission.csv (5000, 2)


In [6]:


# ----------------------------
# 1. COSTANTI
# ----------------------------

if os.path.exists("/content/train.jsonl"):
    base_path = "/content/"
else:
    base_path = "./"

print("Percorso dati:", base_path)

type_df = pd.read_csv(
    os.path.join(base_path, "type_chart_gen2.csv")
)

# Normalizza i nomi dei tipi nella type chart.
type_df["attacking_type"] = (
    type_df["attacking_type"].astype(str).str.lower()
)
type_df["defending_type"] = (
    type_df["defending_type"].astype(str).str.lower()
)

TYPE_MULTIPLIER = {
    (row.attacking_type, row.defending_type): float(row.multiplier)
    for row in type_df.itertuples()
}

PHYSICAL_TYPES = {
    "normal", "fighting", "flying", "poison", "ground",
    "rock", "bug", "ghost", "steel"
}
SPECIAL_TYPES = {
    "fire", "water", "grass", "electric",
    "psychic", "ice", "dragon", "dark"
}

STAT_KEYS = [
    "base_hp", "base_atk", "base_def",
    "base_spa", "base_spd", "base_spe"
]

STAT_SHORT = ["hp", "atk", "def", "spa", "spd", "spe"]

STATUS_LIST = ["slp", "frz", "par", "brn", "tox"]
IMPORTANT_ITEMS = [
    "leftovers", "thickclub", "lightball", "berserkgene",
    "mintberry", "miracleberry", "bitterberry", "przcureberry",
    "quickclaw", "kingsrock", "focusband", "metalpowder",
    "stick"
]

STRATEGIC_MOVES = {
    "curse", "rest", "sleeptalk", "spikes", "rapidspin",
    "explosion", "selfdestruct", "self-destruct", "toxic",
    "thunderwave", "stunspore", "roar", "whirlwind",
    "reflect", "lightscreen", "light-screen", "protect",
    "substitute", "bellydrum", "swordsdance", "recover",
    "softboiled", "moonlight", "morningsun", "synthesis",
    "perishsong", "encore", "batonpass", "haze",
    "sandstorm", "raindance", "sunnyday"
}

# Indicatori strategici basati su specie comunemente rilevanti in GSC OU.
OU_SPECIES = {
    "snorlax", "zapdos", "raikou", "skarmory", "cloyster",
    "tyranitar", "marowak", "umbreon", "suicune", "exeggutor",
    "machamp", "nidoking", "heracross", "steelix", "starmie",
    "forretress", "blissey", "miltank", "gengar"
}


# ----------------------------
# 2. FUNZIONI DI SUPPORTO
# ----------------------------

def safe_float(value, default=0.0):
    try:
        if value is None:
            return default
        value = float(value)
        if not np.isfinite(value):
            return default
        return value
    except (TypeError, ValueError):
        return default


def clean_type_list(types):
    if not isinstance(types, list):
        return []
    return [
        str(t).lower()
        for t in types
        if t is not None and str(t).lower() not in ("notype", "")
    ]


def get_type_multiplier(attack_type, defender_types):
    """Moltiplicatore Gen 2 per una mossa contro uno o due tipi."""
    attack_type = str(attack_type).lower()
    multiplier = 1.0

    for defender_type in clean_type_list(defender_types):
        multiplier *= TYPE_MULTIPLIER.get(
            (attack_type, defender_type), 1.0
        )

    return multiplier


def get_bst(pokemon):
    return sum(safe_float(pokemon.get(k, 0)) for k in STAT_KEYS)


def get_state(turn, player):
    state = turn.get(f"{player}_pokemon_state", {})
    return state if isinstance(state, dict) else {}


def get_move(turn, player):
    move = turn.get(f"{player}_move_details")
    return move if isinstance(move, dict) else None


def get_boost_sum(state):
    boosts = state.get("boosts", {})
    if not isinstance(boosts, dict):
        return 0.0
    return sum(safe_float(v) for v in boosts.values())


def hp_value(state):
    return safe_float(state.get("hp_pct", 0.0), 0.0)


def get_name(state):
    return str(state.get("name", "")).lower()


def count_unique_fainted(timeline, player):
    names = set()
    for turn in timeline:
        state = get_state(turn, player)
        if state.get("status") == "fnt" and get_name(state):
            names.add(get_name(state))
    return len(names)


def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                records.append(json.loads(line))
    return records


# ----------------------------
# 2b. DATABASE SPECIE + SQUADRA P2 OSSERVATA
# ----------------------------

CACHE_DIR = os.path.join(base_path, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

SPECIES_DB = {}


def build_species_db():
    """Dizionario specie -> tipi e stat base, da P1 team e P2 lead (train+test)."""
    cache_path = os.path.join(CACHE_DIR, "species_db.pkl")
    if os.path.exists(cache_path):
        return pd.read_pickle(cache_path)

    db = {}
    for split in ["train", "test"]:
        for battle in load_jsonl(os.path.join(base_path, f"{split}.jsonl")):
            members = list(battle.get("p1_team_details", []) or [])
            lead = battle.get("p2_lead_details")
            if isinstance(lead, dict) and lead:
                members.append(lead)
            for p in members:
                name = str(p.get("name", "")).lower()
                if name and name not in db:
                    db[name] = {
                        "name": name,
                        "types": p.get("types", []),
                        **{k: p.get(k, 0) for k in STAT_KEYS},
                        "item": "unknown",  # l'item di P2 non è noto
                    }
    pd.to_pickle(db, cache_path)
    return db


def get_p2_observed_team(timeline, p2_lead):
    """Squadra di P2 ricostruita: lead + Pokémon comparsi nella timeline."""
    lead_name = str(p2_lead.get("name", "")).lower() if p2_lead else ""
    names = [lead_name] if lead_name else []
    for t in timeline:
        n = get_name(get_state(t, "p2"))
        if n and n not in names:
            names.append(n)

    team = []
    for n in names:
        if n == lead_name:
            team.append(p2_lead)
        elif n in SPECIES_DB:
            team.append(SPECIES_DB[n])
    return team


def avg_best_mult(move_types, defenders):
    """Media, sui difensori, del miglior moltiplicatore tra i tipi di mossa."""
    vals = []
    for d in defenders:
        d_types = clean_type_list(d.get("types", []))
        mults = [get_type_multiplier(t, d_types) for t in move_types]
        vals.append(max(mults) if mults else 1.0)
    return float(np.mean(vals)) if vals else 1.0


def team_hp_left(states, hp_scale, team_total=6):
    """HP residuo stimato della squadra: ultimo HP visto per ogni Pokémon
    rivelato (0 se KO) + 1.0 per ogni Pokémon mai comparso."""
    last = {}
    for s in states:
        n = get_name(s)
        if not n:
            continue
        last[n] = 0.0 if s.get("status") == "fnt" else hp_value(s) / hp_scale
    return sum(last.values()) + max(0, team_total - len(last))


def turn_effectiveness(timeline, atk_player, def_player):
    """Moltiplicatori di tipo reali delle mosse offensive usate, contro il
    Pokémon che stava effettivamente difendendo in quel turno."""
    mults = []
    for t in timeline:
        m = get_move(t, atk_player)
        if not m or safe_float(m.get("base_power", 0)) <= 0:
            continue
        d = SPECIES_DB.get(get_name(get_state(t, def_player)))
        if not d:
            continue
        mults.append(get_type_multiplier(m.get("type", ""), d.get("types", [])))
    return mults


def add_team_features(feat, team, prefix):
    """Statistiche, tipi, strumenti e ruoli della squadra conosciuta."""
    team = team if isinstance(team, list) else []

    feat[f"{prefix}_team_size"] = len(team)

    for stat_key, short in zip(STAT_KEYS, STAT_SHORT):
        values = [safe_float(p.get(stat_key, 0)) for p in team]
        feat[f"{prefix}_sum_{short}"] = sum(values)
        feat[f"{prefix}_mean_{short}"] = np.mean(values) if values else 0
        feat[f"{prefix}_max_{short}"] = max(values) if values else 0
        feat[f"{prefix}_min_{short}"] = min(values) if values else 0
        feat[f"{prefix}_std_{short}"] = np.std(values) if values else 0

    bsts = [get_bst(p) for p in team]
    feat[f"{prefix}_bst_sum"] = sum(bsts)
    feat[f"{prefix}_bst_mean"] = np.mean(bsts) if bsts else 0
    feat[f"{prefix}_bst_max"] = max(bsts) if bsts else 0

    # Profili statistici della squadra.
    atk = [safe_float(p.get("base_atk", 0)) for p in team]
    spa = [safe_float(p.get("base_spa", 0)) for p in team]
    deff = [safe_float(p.get("base_def", 0)) for p in team]
    spd = [safe_float(p.get("base_spd", 0)) for p in team]
    spe = [safe_float(p.get("base_spe", 0)) for p in team]
    hp = [safe_float(p.get("base_hp", 0)) for p in team]
    feat[f"{prefix}_physical_offense_mean"] = float(np.mean(atk)) if atk else 0
    feat[f"{prefix}_special_offense_mean"] = float(np.mean(spa)) if spa else 0
    feat[f"{prefix}_physical_bulk_mean"] = float(np.mean(deff)) if deff else 0
    feat[f"{prefix}_special_bulk_mean"] = float(np.mean(spd)) if spd else 0
    feat[f"{prefix}_speed_mean"] = float(np.mean(spe)) if spe else 0
    feat[f"{prefix}_speed_max"] = max(spe) if spe else 0
    feat[f"{prefix}_fast_count_100"] = sum(v >= 100 for v in spe)
    feat[f"{prefix}_fast_count_80"] = sum(v >= 80 for v in spe)
    feat[f"{prefix}_physical_bias"] = (
        float(np.mean(atk)) - float(np.mean(spa)) if atk and spa else 0
    )
    feat[f"{prefix}_bulk_mean"] = (
        (float(np.mean(hp)) + float(np.mean(deff)) + float(np.mean(spd))) / 3
        if hp and deff and spd else 0
    )

    # Tipi presenti nella squadra.
    all_types = []
    for pokemon in team:
        all_types.extend(clean_type_list(pokemon.get("types", [])))

    for pokemon_type in sorted(
        PHYSICAL_TYPES | SPECIAL_TYPES
    ):
        feat[f"{prefix}_type_count_{pokemon_type}"] = (
            all_types.count(pokemon_type)
        )

    feat[f"{prefix}_unique_types"] = len(set(all_types))
    feat[f"{prefix}_has_steel"] = int("steel" in all_types)
    feat[f"{prefix}_has_dark"] = int("dark" in all_types)
    feat[f"{prefix}_has_ghost"] = int("ghost" in all_types)
    feat[f"{prefix}_has_flying"] = int("flying" in all_types)

    # Strumenti.
    items = [
        str(p.get("item", "") or "").lower()
        for p in team
    ]
    for item in IMPORTANT_ITEMS:
        feat[f"{prefix}_item_{item}"] = items.count(item)

    feat[f"{prefix}_no_item_count"] = sum(
        item in ("", "noitem", "none") for item in items
    )

    # Pokémon e combinazioni note del metagame.
    names = [str(p.get("name", "")).lower() for p in team]
    for species in sorted(OU_SPECIES):
        feat[f"{prefix}_species_{species}"] = int(species in names)

    feat[f"{prefix}_has_marowak_thickclub"] = int(
        any(
            str(p.get("name", "")).lower() == "marowak"
            and str(p.get("item", "")).lower() == "thickclub"
            for p in team
        )
    )
    feat[f"{prefix}_has_pikachu_lightball"] = int(
        any(
            str(p.get("name", "")).lower() == "pikachu"
            and str(p.get("item", "")).lower() == "lightball"
            for p in team
        )
    )


# ----------------------------
# 3. FEATURE ENGINEERING
# ----------------------------

def extract_features(data):
    rows = []

    for battle in data:
        feat = {}
        feat["battle_id"] = battle.get("battle_id")

        p1_team = battle.get("p1_team_details", [])
        if not isinstance(p1_team, list):
            p1_team = []

        p1_lead = p1_team[0] if p1_team else {}
        p2_lead = battle.get("p2_lead_details", {})
        if not isinstance(p2_lead, dict):
            p2_lead = {}

        # ----- Squadre e statistiche base -----
        add_team_features(feat, p1_team, "p1")

        # Per P2 il lead è noto con certezza.
        add_team_features(feat, [p2_lead] if p2_lead else [], "p2lead")

        # Statistiche e differenze del lead.
        for stat_key, short in zip(STAT_KEYS, STAT_SHORT):
            p1_value = safe_float(p1_lead.get(stat_key, 0))
            p2_value = safe_float(p2_lead.get(stat_key, 0))
            feat[f"lead_p1_{short}"] = p1_value
            feat[f"lead_p2_{short}"] = p2_value
            feat[f"lead_diff_{short}"] = p1_value - p2_value

        feat["lead_p1_bst"] = get_bst(p1_lead)
        feat["lead_p2_bst"] = get_bst(p2_lead)
        feat["lead_bst_diff"] = (
            feat["lead_p1_bst"] - feat["lead_p2_bst"]
        )
        feat["lead_p1_faster"] = int(
            safe_float(p1_lead.get("base_spe", 0))
            > safe_float(p2_lead.get("base_spe", 0))
        )
        for stat in ["base_atk", "base_def", "base_spa", "base_spd", "base_hp", "base_spe"]:
            a = safe_float(p1_lead.get(stat, 0))
            b = safe_float(p2_lead.get(stat, 0))
            feat[f"lead_ratio_{stat}"] = a / (b + 1.0)
        feat["lead_p1_physical_bias"] = (
            safe_float(p1_lead.get("base_atk", 0)) -
            safe_float(p1_lead.get("base_spa", 0))
        )
        feat["lead_p2_physical_bias"] = (
            safe_float(p2_lead.get("base_atk", 0)) -
            safe_float(p2_lead.get("base_spa", 0))
        )

        # Matchup difensivo dei lead (proxy basata sui tipi).
        p1_types = clean_type_list(p1_lead.get("types", []))
        p2_types = clean_type_list(p2_lead.get("types", []))

        for attack_type in sorted(PHYSICAL_TYPES | SPECIAL_TYPES):
            feat[f"p1lead_def_mult_from_{attack_type}"] = (
                get_type_multiplier(attack_type, p1_types)
            )
            feat[f"p2lead_def_mult_from_{attack_type}"] = (
                get_type_multiplier(attack_type, p2_types)
            )

        p1_best_type_matchup = [
            get_type_multiplier(t, p2_types) for t in p1_types
        ]
        p2_best_type_matchup = [
            get_type_multiplier(t, p1_types) for t in p2_types
        ]
        feat["lead_best_type_attack_p1"] = (
            max(p1_best_type_matchup) if p1_best_type_matchup else 1.0
        )
        feat["lead_best_type_attack_p2"] = (
            max(p2_best_type_matchup) if p2_best_type_matchup else 1.0
        )

        # ----- Timeline -----
        timeline = battle.get("battle_timeline", [])
        if not isinstance(timeline, list):
            timeline = []

        feat["num_turns"] = len(timeline)

        p1_states = [get_state(t, "p1") for t in timeline]
        p2_states = [get_state(t, "p2") for t in timeline]

        p1_hp = [hp_value(s) for s in p1_states]
        p2_hp = [hp_value(s) for s in p2_states]
        hp_diffs = [a - b for a, b in zip(p1_hp, p2_hp)]

        # Snapshot HP a turni prefissati.
        for checkpoint in [1, 3, 5, 10, 15, 20, 25, 30]:
            idx = min(checkpoint - 1, len(timeline) - 1)
            if idx >= 0:
                feat[f"hp_diff_turn_{checkpoint}"] = hp_diffs[idx]
                feat[f"p1_hp_turn_{checkpoint}"] = p1_hp[idx]
                feat[f"p2_hp_turn_{checkpoint}"] = p2_hp[idx]
            else:
                feat[f"hp_diff_turn_{checkpoint}"] = 0
                feat[f"p1_hp_turn_{checkpoint}"] = 0
                feat[f"p2_hp_turn_{checkpoint}"] = 0

        if timeline:
            feat["p1_last_active_hp"] = p1_hp[-1]
            feat["p2_last_active_hp"] = p2_hp[-1]
            feat["last_active_hp_diff"] = hp_diffs[-1]
            feat["p1_last_active_fainted"] = int(
                p1_states[-1].get("status") == "fnt"
            )
            feat["p2_last_active_fainted"] = int(
                p2_states[-1].get("status") == "fnt"
            )
        else:
            for key in [
                "p1_last_active_hp", "p2_last_active_hp",
                "last_active_hp_diff", "p1_last_active_fainted",
                "p2_last_active_fainted"
            ]:
                feat[key] = 0

        # HP trend: regressione lineare sul differenziale osservato.
        if len(hp_diffs) >= 2:
            feat["hp_diff_trend"] = float(
                np.polyfit(np.arange(len(hp_diffs)), hp_diffs, 1)[0]
            )
            feat["hp_diff_mean"] = float(np.mean(hp_diffs))
            feat["hp_diff_min"] = float(np.min(hp_diffs))
            feat["hp_diff_max"] = float(np.max(hp_diffs))
            feat["hp_diff_std"] = float(np.std(hp_diffs))
        else:
            feat["hp_diff_trend"] = 0
            feat["hp_diff_mean"] = 0
            feat["hp_diff_min"] = 0
            feat["hp_diff_max"] = 0
            feat["hp_diff_std"] = 0

        feat["p1_active_hp_change"] = (
            p1_hp[-1] - p1_hp[0] if len(p1_hp) >= 2 else 0
        )
        feat["p2_active_hp_change"] = (
            p2_hp[-1] - p2_hp[0] if len(p2_hp) >= 2 else 0
        )

        # Fasi della lotta.
        for phase, start, end in [
            ("early", 0, 5),
            ("mid", 5, 15),
            ("late", 15, 30)
        ]:
            diffs = hp_diffs[start:end]
            feat[f"{phase}_hp_diff_mean"] = (
                float(np.mean(diffs)) if diffs else 0
            )
            feat[f"{phase}_hp_diff_last"] = (
                float(diffs[-1]) if diffs else 0
            )

        # KO distinti osservati.
        feat["p1_observed_fainted_count"] = count_unique_fainted(
            timeline, "p1"
        )
        feat["p2_observed_fainted_count"] = count_unique_fainted(
            timeline, "p2"
        )
        feat["observed_fainted_diff"] = (
            feat["p2_observed_fainted_count"]
            - feat["p1_observed_fainted_count"]
        )

        # Pokémon attivi distinti e cambi.
        p1_names = [get_name(s) for s in p1_states if get_name(s)]
        p2_names = [get_name(s) for s in p2_states if get_name(s)]

        feat["p1_distinct_active_pokemon"] = len(set(p1_names))
        feat["p2_distinct_active_pokemon"] = len(set(p2_names))

        feat["p1_switches_observed"] = sum(
            a != b for a, b in zip(p1_names[:-1], p1_names[1:])
        )
        feat["p2_switches_observed"] = sum(
            a != b for a, b in zip(p2_names[:-1], p2_names[1:])
        )

        # Status e boost.
        for player, states in [("p1", p1_states), ("p2", p2_states)]:
            for status in STATUS_LIST:
                status_states = [
                    s for s in states if s.get("status") == status
                ]
                feat[f"{player}_{status}_turns"] = len(status_states)
                feat[f"{player}_{status}_pokemon_count"] = len({
                    get_name(s) for s in status_states if get_name(s)
                })

            feat[f"{player}_boost_sum_mean"] = (
                np.mean([get_boost_sum(s) for s in states])
                if states else 0
            )
            feat[f"{player}_boost_sum_max"] = (
                max([get_boost_sum(s) for s in states], default=0)
            )
            feat[f"{player}_boost_sum_last"] = (
                get_boost_sum(states[-1]) if states else 0
            )

        feat["boost_last_diff"] = (
            feat["p1_boost_sum_last"] - feat["p2_boost_sum_last"]
        )

        # Mosse osservate.
        p1_moves = [get_move(t, "p1") for t in timeline]
        p2_moves = [get_move(t, "p2") for t in timeline]
        p1_moves = [m for m in p1_moves if m is not None]
        p2_moves = [m for m in p2_moves if m is not None]

        for player, moves in [("p1", p1_moves), ("p2", p2_moves)]:
            names = [
                str(m.get("name", "")).lower().replace(" ", "")
                for m in moves
            ]
            powers = [
                safe_float(m.get("base_power", 0))
                for m in moves
            ]

            feat[f"{player}_move_count"] = len(moves)
            feat[f"{player}_avg_move_power"] = (
                float(np.mean(powers)) if powers else 0
            )
            feat[f"{player}_max_move_power"] = (
                max(powers) if powers else 0
            )
            feat[f"{player}_high_power_moves"] = sum(
                power >= 100 for power in powers
            )

            for move_name in sorted(STRATEGIC_MOVES):
                normalized = move_name.replace("-", "").replace(" ", "")
                feat[f"{player}_move_{normalized}"] = sum(
                    n == normalized for n in names
                )

            feat[f"{player}_explosion_count"] = sum(
                n in ("explosion", "selfdestruct") for n in names
            )
            feat[f"{player}_rest_count"] = names.count("rest")
            feat[f"{player}_sleeptalk_count"] = names.count("sleeptalk")
            feat[f"{player}_spikes_count"] = names.count("spikes")
            feat[f"{player}_rapidspin_count"] = names.count("rapidspin")
            feat[f"{player}_curse_count"] = names.count("curse")
            feat[f"{player}_toxic_count"] = names.count("toxic")
            feat[f"{player}_thunderwave_count"] = names.count("thunderwave")

            # Gen 2: fisico/speciale dipende dal TIPO.
            physical = 0
            special = 0
            status_moves = 0
            for move in moves:
                category = str(move.get("category", "")).lower()
                move_type = str(move.get("type", "")).lower()
                power = safe_float(move.get("base_power", 0))

                if category in ("status", "non-damaging") or power <= 0:
                    status_moves += 1
                elif move_type in PHYSICAL_TYPES:
                    physical += 1
                elif move_type in SPECIAL_TYPES:
                    special += 1

            feat[f"{player}_physical_moves"] = physical
            feat[f"{player}_special_moves"] = special
            feat[f"{player}_status_moves"] = status_moves

        # RestTalk osservato.
        feat["p1_resttalk_combo"] = int(
            feat["p1_rest_count"] > 0
            and feat["p1_sleeptalk_count"] > 0
        )
        feat["p2_resttalk_combo"] = int(
            feat["p2_rest_count"] > 0
            and feat["p2_sleeptalk_count"] > 0
        )

        # Copertura offensiva osservata (lead vs lead).
        p1_lead_move_types = {
            str(m.get("type", "")).lower()
            for m in p1_moves
            if str(m.get("type", "")).lower() in (PHYSICAL_TYPES | SPECIAL_TYPES)
        }
        p2_lead_move_types = {
            str(m.get("type", "")).lower()
            for m in p2_moves
            if str(m.get("type", "")).lower() in (PHYSICAL_TYPES | SPECIAL_TYPES)
        }

        feat["p1_move_type_coverage_vs_p2lead"] = max(
            [
                get_type_multiplier(t, p2_types)
                for t in p1_lead_move_types
            ] or [1.0]
        )
        feat["p2_move_type_coverage_vs_p1lead"] = max(
            [
                get_type_multiplier(t, p1_types)
                for t in p2_lead_move_types
            ] or [1.0]
        )

        # Archetipi dedotti dalle mosse osservate.
        p1_move_set = {
            str(m.get("name", "")).lower().replace(" ", "")
            for m in p1_moves
        }
        p2_move_set = {
            str(m.get("name", "")).lower().replace(" ", "")
            for m in p2_moves
        }

        for player, move_set in [
            ("p1", p1_move_set), ("p2", p2_move_set)
        ]:
            feat[f"{player}_observed_curse_rest"] = int(
                "curse" in move_set and "rest" in move_set
            )
            feat[f"{player}_observed_spikes_spin"] = int(
                "spikes" in move_set and "rapidspin" in move_set
            )
            feat[f"{player}_observed_screen"] = int(
                "reflect" in move_set
                or "lightscreen" in move_set
                or "light-screen" in move_set
            )

        # ----- NUOVO: squadra P2 osservata (lead + Pokémon rivelati) -----
        p2_team_obs = get_p2_observed_team(timeline, p2_lead)
        add_team_features(feat, p2_team_obs, "p2obs")
        feat["p2obs_revealed_count"] = len(p2_team_obs)

        # Differenze P1 (squadra intera) vs P2 (squadra osservata).
        for short in ["atk", "def", "spa", "spd", "spe", "hp"]:
            feat[f"obs_mean_diff_{short}"] = (
                feat[f"p1_mean_{short}"] - feat[f"p2obs_mean_{short}"]
            )
        feat["obs_bst_mean_diff"] = feat["p1_bst_mean"] - feat["p2obs_bst_mean"]
        feat["obs_speed_max_diff"] = feat["p1_speed_max"] - feat["p2obs_speed_max"]

        # Copertura delle mosse contro TUTTI i Pokémon rivelati / la squadra P1.
        p1_move_types_all = {
            str(m.get("type", "")).lower() for m in p1_moves
        } & (PHYSICAL_TYPES | SPECIAL_TYPES)
        p2_move_types_all = {
            str(m.get("type", "")).lower() for m in p2_moves
        } & (PHYSICAL_TYPES | SPECIAL_TYPES)

        feat["p1_moves_vs_p2obs_avg_best"] = avg_best_mult(
            p1_move_types_all, p2_team_obs
        )
        feat["p2_moves_vs_p1team_avg_best"] = avg_best_mult(
            p2_move_types_all, p1_team
        )
        feat["coverage_team_diff"] = (
            feat["p1_moves_vs_p2obs_avg_best"]
            - feat["p2_moves_vs_p1team_avg_best"]
        )

        # Debolezze di tipo di P1 rispetto ai tipi dei P2 rivelati.
        p2_obs_types = set()
        for p in p2_team_obs:
            p2_obs_types |= set(clean_type_list(p.get("types", [])))
        feat["p1_team_avg_mult_vs_p2obs_types"] = (
            float(np.mean([
                get_type_multiplier(t, p.get("types", []))
                for p in p1_team for t in p2_obs_types
            ])) if p1_team and p2_obs_types else 1.0
        )

        # ----- NUOVO v6: HP residuo stimato delle squadre -----
        all_hp = p1_hp + p2_hp
        hp_scale = 100.0 if (all_hp and max(all_hp) > 1.0) else 1.0
        feat["p1_team_hp_left"] = team_hp_left(p1_states, hp_scale)
        feat["p2_team_hp_left"] = team_hp_left(p2_states, hp_scale)
        feat["team_hp_left_diff"] = (
            feat["p1_team_hp_left"] - feat["p2_team_hp_left"]
        )
        feat["team_hp_left_ratio"] = (
            feat["p1_team_hp_left"] / (feat["p2_team_hp_left"] + 0.5)
        )
        feat["p1_alive_est"] = 6 - feat["p1_observed_fainted_count"]
        feat["p2_alive_est"] = 6 - feat["p2_observed_fainted_count"]
        feat["alive_est_diff"] = feat["p1_alive_est"] - feat["p2_alive_est"]

        # ----- NUOVO v6: efficacia reale delle mosse turno per turno -----
        for player, other in [("p1", "p2"), ("p2", "p1")]:
            mults = turn_effectiveness(timeline, player, other)
            feat[f"{player}_eff_mean"] = float(np.mean(mults)) if mults else 1.0
            feat[f"{player}_eff_super_count"] = sum(m > 1 for m in mults)
            feat[f"{player}_eff_resisted_count"] = sum(0 < m < 1 for m in mults)
            feat[f"{player}_eff_immune_count"] = sum(m == 0 for m in mults)
            feat[f"{player}_eff_n"] = len(mults)
        feat["eff_mean_diff"] = feat["p1_eff_mean"] - feat["p2_eff_mean"]
        feat["eff_super_diff"] = (
            feat["p1_eff_super_count"] - feat["p2_eff_super_count"]
        )
        feat["eff_resisted_diff"] = (
            feat["p1_eff_resisted_count"] - feat["p2_eff_resisted_count"]
        )

        # Etichetta soltanto nel train.
        if "player_won" in battle:
            feat["player_won"] = int(bool(battle["player_won"]))

        rows.append(feat)

    return pd.DataFrame(rows).replace(
        [np.inf, -np.inf], np.nan
    ).fillna(0)


# ----------------------------
# CONFIGURAZIONE
# ----------------------------

QUICK = True              # True: 1 modello, 3 fold (per iterare sulle feature)
                          # False: XGB + Hist + LogReg, 5 fold, media semplice
FEATURE_VERSION = "v6"    # CAMBIALO ogni volta che modifichi extract_features
RANDOM_STATE = 42
N_SPLITS = 3 if QUICK else 5


# ----------------------------
# 4. CARICAMENTO DATI (con cache delle feature)
# ----------------------------

def load_features(split):
    cache_path = os.path.join(CACHE_DIR, f"{split}_features_{FEATURE_VERSION}.pkl")
    if os.path.exists(cache_path):
        print(f"Feature {split} caricate dalla cache: {cache_path}")
        return pd.read_pickle(cache_path)

    print(f"Estrazione feature {split}...")
    data = load_jsonl(os.path.join(base_path, f"{split}.jsonl"))
    df = extract_features(data)
    del data
    df.to_pickle(cache_path)
    return df


SPECIES_DB = build_species_db()
print("Specie nel database:", len(SPECIES_DB))

train_df = load_features("train")
test_df = load_features("test")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


# ----------------------------
# 5. MATRICE FEATURE
# ----------------------------

target = "player_won"
id_col = "battle_id"

features = [c for c in train_df.columns if c not in [target, id_col]]

X_train = train_df[features].copy()
y_train = train_df[target].astype(int).copy()
X_test = test_df.reindex(columns=features, fill_value=0).copy()

print("Numero feature:", len(features))
print("Distribuzione target:")
print(y_train.value_counts(normalize=True))


# ----------------------------
# 6. MODELLI
# ----------------------------

def make_xgb():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        XGBClassifier(
            n_estimators=700, max_depth=3, learning_rate=0.03,
            subsample=0.85, colsample_bytree=0.85,
            min_child_weight=5, reg_lambda=8, reg_alpha=0.2, gamma=0.05,
            eval_metric="logloss", objective="binary:logistic",
            n_jobs=-1, random_state=RANDOM_STATE
        )
    )


def make_hist():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        HistGradientBoostingClassifier(
            max_iter=400, learning_rate=0.03, max_leaf_nodes=15,
            l2_regularization=2.0, random_state=RANDOM_STATE
        )
    )


def make_logreg():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(
            C=0.5, max_iter=3000, solver="lbfgs",
            random_state=RANDOM_STATE
        )
    )


if QUICK:
    model_factories = {"XGBoost": make_xgb}
else:
    model_factories = {
        "XGBoost": make_xgb,
        "HistGradientBoosting": make_hist,
        "Logistic Regression": make_logreg,
    }

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
folds = list(skf.split(X_train, y_train))


def evaluate_model(model_name, model_factory):
    """OOF + media delle predizioni test sui fold, con cache su disco."""
    safe_name = model_name.replace(" ", "_")
    cache_path = os.path.join(
        CACHE_DIR, f"pred_{FEATURE_VERSION}_{safe_name}_k{N_SPLITS}.npz"
    )

    if os.path.exists(cache_path):
        print(f"  Predizioni {model_name} caricate dalla cache")
        saved = np.load(cache_path)
        return saved["oof"], saved["test"]

    oof = np.zeros(len(X_train), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
        model = model_factory()
        model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        valid_prob = model.predict_proba(X_train.iloc[va_idx])[:, 1]
        oof[va_idx] = valid_prob
        test_pred += model.predict_proba(X_test)[:, 1] / N_SPLITS

        fold_acc = accuracy_score(
            y_train.iloc[va_idx], (valid_prob >= 0.5).astype(int)
        )
        print(f"  Fold {fold}: accuracy = {fold_acc:.5f}")

    np.savez(cache_path, oof=oof, test=test_pred)
    return oof, test_pred


results = {}
oof_predictions = {}
test_predictions = {}

for name, factory in model_factories.items():
    print("\n" + "=" * 60 + "\nMODELLO:", name)
    oof, test_pred = evaluate_model(name, factory)
    results[name] = accuracy_score(y_train, (oof >= 0.5).astype(int))
    oof_predictions[name] = oof
    test_predictions[name] = test_pred
    print(f"  OOF accuracy {name}: {results[name]:.5f}")


# ----------------------------
# 7. ENSEMBLE: media semplice, soglia 0.5
# ----------------------------

print("\n========== RISULTATI ==========")
for name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name}: {score:.5f}")

if len(model_factories) > 1:
    ens_oof = np.mean([oof_predictions[n] for n in model_factories], axis=0)
    ens_test = np.mean([test_predictions[n] for n in model_factories], axis=0)
    ens_acc = accuracy_score(y_train, (ens_oof >= 0.5).astype(int))
    print(f"\nEnsemble (media semplice, soglia 0.5): {ens_acc:.5f}")
    final_test_predictions = ens_test
else:
    only = next(iter(model_factories))
    final_test_predictions = test_predictions[only]


# ----------------------------
# 8. SUBMISSION
# ----------------------------

sample_sub = pd.read_csv(os.path.join(base_path, "sample_submission.csv"))

preds_series = pd.Series(
    (final_test_predictions >= 0.5).astype(int),
    index=test_df[id_col]
)

submission = sample_sub[[id_col]].copy()
submission[target] = submission[id_col].map(preds_series).astype(int)

assert list(submission.columns) == ["battle_id", "player_won"]
assert len(submission) == len(test_df)
assert set(submission["player_won"].unique()).issubset({0, 1})

submission.to_csv("submission.csv", index=False)
print("\nSubmission creata: submission.csv", submission.shape)

Percorso dati: /content/
Specie nel database: 247
Estrazione feature train...
Estrazione feature test...
Train shape: (10000, 581)
Test shape: (5000, 580)
Numero feature: 579
Distribuzione target:
player_won
0    0.5
1    0.5
Name: proportion, dtype: float64

MODELLO: XGBoost
  Fold 1: accuracy = 0.65957
  Fold 2: accuracy = 0.68287
  Fold 3: accuracy = 0.66697
  OOF accuracy XGBoost: 0.66980

========== RISULTATI ==========
XGBoost: 0.66980

Submission creata: submission.csv (5000, 2)
